# London vs. West Mercia

Both forces have now been analysed independently:
`notebooks/london/01-05` and `notebooks/west-mercia/01-05`. This notebook
puts them side by side. Everything here is a **direct comparison of
already-established numbers** from those two threads — no new cleaning
decisions, just combining what we already trust.

Both datasets cover the same 13 months (May 2025 - May 2026) and use the
same ONS mid-2024 population and MHCLG IMD 2019 sources, so a direct
comparison is fair.

In [ ]:
import sys
sys.path.append("../../src")

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
import seaborn as sns

from load_data import load_force_data, load_population, load_deprivation, load_lad_boundaries
from clean import clean_crime_data, add_area_column, LONDON_BOROUGHS, WEST_MERCIA_DISTRICTS

sns.set_theme(style="whitegrid")

# Two consistent colors for "London" vs "West Mercia" used across every
# chart in this notebook -- picking them once here means every chart's
# legend agrees with every other chart's, which matters when a reader
# flips between them.
LONDON_COLOR = "#4C72B0"
WM_COLOR = "#DD8452"

## Building one summary table per force

Both London's and West Mercia's notebooks build the same shape of table
(area, crime count, population, IMD score) via the same generic loaders --
so rather than importing five notebooks' worth of code, we just rebuild
that one small table twice, once per force, with a helper function.

In [ ]:
def build_summary(force: str, areas: set[str], area_column: str) -> pd.DataFrame:
    """One row per area (borough/district): crime count, population, IMD score."""
    df = clean_crime_data(load_force_data(force))
    df = add_area_column(df, column_name=area_column)
    df = df[df[area_column].isin(areas)]

    pop = load_population(areas, name_column=area_column)
    dep = load_deprivation(areas, name_column=area_column)
    counts = df.groupby(area_column).size().rename("Crimes").reset_index()

    merged = (
        counts
        .merge(pop, on=area_column, validate="one_to_one")
        .merge(dep, on=area_column, validate="one_to_one")
    )
    merged["rate_per_1000"] = merged["Crimes"] / merged["Population"] * 1000
    merged["Force"] = force
    return merged.rename(columns={area_column: "Area"})


london_summary = build_summary("london", LONDON_BOROUGHS, "Borough")
wm_summary = build_summary("west-mercia", WEST_MERCIA_DISTRICTS, "District")

london_summary.shape, wm_summary.shape

## Headline: overall per-capita rate

For a single force-wide number, we don't need to go through boroughs at
all -- just total crimes (every row, including the ones with no LSOA/
borough match) divided by total population. This uses more of the data
than summing the per-borough table would (that table excludes unlocated
rows), so it's the more complete version of this one number.

In [ ]:
london_all = clean_crime_data(load_force_data("london"))
wm_all = clean_crime_data(load_force_data("west-mercia"))

london_pop_total = load_population(LONDON_BOROUGHS, name_column="Borough")["Population"].sum()
wm_pop_total = load_population(WEST_MERCIA_DISTRICTS, name_column="District")["Population"].sum()

london_overall_rate = len(london_all) / london_pop_total * 1000
wm_overall_rate = len(wm_all) / wm_pop_total * 1000

print(f"London:       {len(london_all):>9,} crimes / {london_pop_total:>9,} residents "
      f"= {london_overall_rate:.1f} per 1,000/yr")
print(f"West Mercia:  {len(wm_all):>9,} crimes / {wm_pop_total:>9,} residents "
      f"= {wm_overall_rate:.1f} per 1,000/yr")
print(f"\nLondon's overall rate is {london_overall_rate / wm_overall_rate:.2f}x West Mercia's.")

London's overall per-capita rate is about **65% higher** than West
Mercia's. The next chart asks: is that uniformly true across every crime
type, or concentrated in a few?

## Chart — per-capita rate by crime type

Trimmed to the **top 6 crime types by combined total** (London + West
Mercia together), for the same readability reason as both forces'
individual notebook 02 heatmaps. Using the *combined* ranking here (rather
than each force's own top 6, which differ slightly) matters: London's and
West Mercia's own top-6 lists aren't identical, and using each
independently could have silently dropped "Theft from the person" from
West Mercia's side of the chart — which would have hidden the very
category that turns out to matter most below.

In [ ]:
london_rate_by_type = (london_all["Crime type"].value_counts() / london_pop_total * 1000).rename("London")
wm_rate_by_type = (wm_all["Crime type"].value_counts() / wm_pop_total * 1000).rename("West Mercia")
rate_by_type = pd.concat([london_rate_by_type, wm_rate_by_type], axis=1)

# Rank by the COMBINED raw count across both forces (not each force's own
# rate), so the same 6 categories appear on both sides of the chart --
# see the markdown above for why that matters here specifically.
combined_counts = london_all["Crime type"].value_counts().add(
    wm_all["Crime type"].value_counts(), fill_value=0
)
top_crime_types = combined_counts.sort_values(ascending=False).index[:6]
rate_by_type = rate_by_type.loc[top_crime_types].sort_values("London")

y = np.arange(len(rate_by_type))
bar_height = 0.38

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(y + bar_height / 2, rate_by_type["London"], height=bar_height, color=LONDON_COLOR, label="London")
ax.barh(y - bar_height / 2, rate_by_type["West Mercia"], height=bar_height, color=WM_COLOR, label="West Mercia")
ax.set_yticks(y)
ax.set_yticklabels(rate_by_type.index)
ax.set_xlabel("Recorded crimes per 1,000 residents (annual)")
ax.set_title("Per-capita crime rate by type (top 6 combined): London vs. West Mercia")
ax.legend()
fig.tight_layout()

### The headline finding of this whole comparison

**"Violence and sexual offences" -- the single largest crime category in
both forces -- occurs at almost exactly the same per-capita rate: 33.2 per
1,000/yr in London vs. 32.8 in West Mercia (about a 1% difference).**
London's overall 65%-higher rate comes almost entirely from other
categories visible in the chart above: **theft from the person is ~35x
higher in London**, vehicle crime ~3.7x, anti-social behaviour ~2.1x.
(Two more categories tell the same story but fell outside this top-6-by-
volume chart: robbery is ~5x higher in London, drugs ~3x — both real,
just smaller in absolute terms than the six shown here.)

In other words: **London isn't uniformly "more dangerous"** — it has a
much higher rate of opportunistic, dense-crowd/urban crime (pickpocketing,
vehicle crime, street drug activity), while the more serious violent-crime
rate per resident is close to identical to a much smaller, more rural force
area. This is exactly the kind of nuance that total counts or even a
single "overall rate" number would have hidden.

## Chart — monthly per-capita trend

In [ ]:
london_monthly = london_all.groupby("Month").size().sort_index() / london_pop_total * 1000
wm_monthly = wm_all.groupby("Month").size().sort_index() / wm_pop_total * 1000

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(london_monthly.index, london_monthly.values, marker="o", color=LONDON_COLOR, label="London")
ax.plot(wm_monthly.index, wm_monthly.values, marker="o", color=WM_COLOR, label="West Mercia")
ax.set_ylabel("Crimes per 1,000 residents (monthly)")
ax.set_title("Monthly per-capita crime rate: London vs. West Mercia")
ax.legend()
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout()

Putting both on **one shared y-axis** (rather than two separate axes) is a
deliberate choice: a dual-axis chart could be stretched to make the two
lines' *shapes* look however similar or different you wanted, which would
misrepresent the real, large gap in absolute level. On a single axis, both
the gap and the shape are honestly comparable at once.

Both lines follow a strikingly similar shape (peak in July 2025, trough in
February 2026) despite the level being very different — good evidence that
there's genuine national seasonality in recorded crime, not just noise in
one force's data.

## Chart — deprivation vs. crime rate, both forces together

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(london_summary["IMD_score"], london_summary["rate_per_1000"],
           color=LONDON_COLOR, s=45, label="London (33 boroughs)")
ax.scatter(wm_summary["IMD_score"], wm_summary["rate_per_1000"],
           color=WM_COLOR, s=45, label="West Mercia (9 districts)")
ax.set_xlabel("IMD 2019 average score (higher = more deprived)")
ax.set_ylabel("Recorded crimes per 1,000 residents (annual)")
ax.set_title("Deprivation vs. per-capita crime rate: London vs. West Mercia")
ax.legend()
fig.tight_layout()

West Mercia's districts (orange) cover a similar deprivation range to many
London boroughs, but sit consistently lower on crime rate — reinforcing
that London's higher overall rate isn't just "explained away" by having
more deprived areas; something about London itself (likely the
urban/footfall effect from notebook 03 in both threads) adds to it on top
of deprivation.

## Chart — side-by-side maps, same color scale

Each force's own `05_map.ipynb` uses a color scale optimised for *that
force's* internal contrast. Here we deliberately use one **shared** scale
across both maps, because the question is different: not "which parts of
West Mercia are relatively higher-crime", but "how does West Mercia's
*entire range* compare to London's". A shared scale answers that honestly;
two independently-scaled maps would make West Mercia look just as
"varied" as London, which isn't true.

In [ ]:
london_boundaries = load_lad_boundaries(LONDON_BOROUGHS, name_column="Borough")
london_geo = london_boundaries.merge(london_summary, left_on="Borough", right_on="Area", validate="one_to_one")

wm_boundaries = load_lad_boundaries(WEST_MERCIA_DISTRICTS, name_column="District")
wm_geo = wm_boundaries.merge(wm_summary, left_on="District", right_on="Area", validate="one_to_one")

# A shared Normalize object, built from BOTH datasets' min/max, is what
# makes the two maps' colors directly comparable -- without it, each
# .plot() call would auto-scale to its own data and silently break the
# comparison.
vmin = min(london_geo["rate_per_1000"].min(), wm_geo["rate_per_1000"].min())
vmax = max(london_geo["rate_per_1000"].max(), wm_geo["rate_per_1000"].max())
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
cmap = "rocket_r"

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
london_geo.plot(column="rate_per_1000", cmap=cmap, norm=norm, linewidth=0.6, edgecolor="white", ax=axes[0])
axes[0].set_title("London")
axes[0].set_axis_off()

wm_geo.plot(column="rate_per_1000", cmap=cmap, norm=norm, linewidth=0.6, edgecolor="white", ax=axes[1])
axes[1].set_title("West Mercia")
axes[1].set_axis_off()

# A single ScalarMappable + one shared colorbar (rather than one per
# subplot) is what tells the reader "these two maps use the same scale".
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes, orientation="horizontal", fraction=0.05, pad=0.02, shrink=0.5)
cbar.set_label("Crimes per 1,000 residents (annual) -- shared scale")
fig.suptitle("Per-capita crime rate: London vs. West Mercia (same color scale)")

West Mercia is uniformly pale next to London on this shared scale — a
direct visual confirmation of the headline number at the top of this
notebook. London's internal variation (Westminster's dark hotspot) is also
clearly visible even at this compressed scale.

## Summary

- London's overall per-capita crime rate is ~65% higher than West
  Mercia's — but this gap is **not uniform across crime types**. Violent
  and sexual offences occur at almost identical per-capita rates in both;
  the gap is concentrated in opportunistic/urban crime (theft from the
  person, vehicle crime, drugs, ASB).
- Both forces show a similar seasonal shape month-to-month, just at
  different levels — suggestive of genuine national seasonality.
- Deprivation correlates with crime rate in both, but the relationship is
  cleaner in West Mercia (no extreme outlier) than in London (Westminster/
  City of London had to be excluded to see a comparable correlation).
  West Mercia's districts also sit *below* London's boroughs at similar
  deprivation levels, suggesting deprivation alone doesn't explain
  London's higher rate.
- Both forces have one district/borough whose crime rate exceeds what its
  deprivation score alone would predict (Westminster, Worcester) — in both
  cases plausibly explained by non-resident footfall (tourism/retail vs. a
  compact city centre), just at very different scales.